# PubChem → Gaussian Input File Pipeline

This notebook automates the generation of **Gaussian input files** (`.com`) and **SLURM submission scripts** (`.sh`) from a list of molecule names. It handles the entire workflow:

1. **Resolve** molecule names → PubChem CIDs + `IsomericSMILES` + properties
2. **Conformer search** (RDKit ETKDGv3 + MMFF94) — embed an ensemble, rank it, and keep the **top 3 lowest-energy distinct** conformers per molecule
3. **Write** one Gaussian `.com` per conformer (opt + freq via Link1)
4. **Generate** SLURM `.sh` scripts for cluster submission

Each step writes a manifest-linked log CSV. **v2.1: every run writes one immutable folder** `runs/<study>/<run_id>/` — there is no resume/append. To add molecules later, start a **new run** (a fresh `run_id`).

> **v2:** the conformer search replaces the v1.1 single-geometry path as the default. **v2.1:** molecules with **undefined stereochemistry are no longer skipped** — they yield **one loudly-flagged PROVISIONAL structure** (arbitrary choice at the undefined centre, `dE=NA`), never silently guessed and never presented as the defined stereoisomer. The legacy Open Babel single-geometry path is preserved as an optional appendix at the bottom.

### Requirements
- Python ≥ 3.11, pandas, requests
- RDKit (`conda install -c conda-forge rdkit`) — conformer search
- The `pipeline/` package (included in this repo)
- Open Babel only if you run the optional v1.1 legacy appendix

---
## 0 · Configuration

Edit the parameters below to match your **cluster**, **level of theory**, and **study label**. Each run creates a fresh, immutable package at `runs/<study>/<run_id>/`: `run_manifest.json` has an immutable `run_id` and is never silently replaced, so re-running is always a new folder rather than an in-place overwrite.

In [ ]:
import os, sys, uuid

# Make sure the pipeline package is importable
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

# ─── Study label + immutable per-run directory (v2.1) ─────────────────
# Every run writes ONE immutable folder: runs/<study>/<run_id>/. There is no
# resume/append — to add molecules later, start a NEW run (a fresh run_id).
STUDY   = "my_study"                  # <-- label this batch, e.g. "nucleobases"
RUN_ID  = str(uuid.uuid4())           # opaque, immutable run identity
RUN_DIR = os.path.join("runs", STUDY, RUN_ID)

# ─── Output files / directories (ALL inside the run folder) ───────────
MOLECULE_CSV  = os.path.join(RUN_DIR, "molecules_pubchem_smiles.csv")
DIAG_CSV      = os.path.join(RUN_DIR, "molecules_pubchem_diagnostics.csv")
CONF_XYZ_DIR  = os.path.join(RUN_DIR, "conformer_xyz")   # one XYZ per kept conformer
CONF_LOG_CSV  = os.path.join(RUN_DIR, "conformer_log.csv")
CONF_FAIL_CSV = os.path.join(RUN_DIR, "conformer_search_failed.csv")
GAUSSIAN_JOBS = os.path.join(RUN_DIR, "gaussian_jobs")   # COM + SH co-located
COM_LOG_CSV   = os.path.join(RUN_DIR, "com_write_log.csv")
SLURM_LOG_CSV = os.path.join(RUN_DIR, "slurm_write_log.csv")
MANIFEST_JSON = os.path.join(RUN_DIR, "run_manifest.json")

# ─── Conformer search parameters (v2) ─────────────────────────────────
# RDKit ETKDGv3 embed (RMSD-pruned) → MMFF94 rank → keep top-N distinct.
N_GENERATE = 20     # conformers embedded per molecule
TOP_N      = 3      # lowest-energy distinct conformers carried to DFT
RMSD_PRUNE = 0.5    # Å; RMSD threshold for duplicate pruning at embed time
SEED       = 42     # fixed random seed → reproducible ensemble

# ─── Gaussian parameters ──────────────────────────────────────────────
NPROC = 16

ROUTE_OPT  = "# opt=(tight,calcfc) b3lyp/6-311++g(2df,2p) scrf=(iefpcm,solvent=water)"
ROUTE_FREQ = "# freq b3lyp/6-311++g(2df,2p) scrf=(iefpcm,solvent=water) temperature=298 Geom=AllChk Guess=Read"
TITLE_SUFFIX = "PCM 298 K 6-311++G(2df,2p)"

# ─── SLURM parameters (edit for your cluster) ─────────────────────────
SLURM_ACCOUNT = "myaccount"   # <-- your allocation
SLURM_CPUS    = 16
SLURM_MEM     = "32G"
SLURM_TIME    = "24:00:00"

# ─── PubChem settings ─────────────────────────────────────────────────
CACHE_DIR = ".pubchem_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

for d in (RUN_DIR, CONF_XYZ_DIR, GAUSSIAN_JOBS):
    os.makedirs(d, exist_ok=True)

print("Working directory:", os.getcwd())
print("Run package:", RUN_DIR)

---
## 1 · Define Your Molecules

**Edit the lists below** for your project. See `examples/nucleobases_nucleosides.py` for a more detailed example with aliases and fallback queries.

### Quick guide
- `molecules`: list of human-readable labels
- `alias`: maps a label → the PubChem query string (needed when the common name differs from what PubChem expects)
- `fallback_queries`: if the primary alias fails, these alternatives are tried in order
- `expected_formulas`: optional — helps the scoring heuristic pick the right PubChem record

In [ ]:
# ── Molecule list ──────────────────────────────────────────────────────
# Replace these with your own molecules.

molecules = [
    "Water",
    "Glycine",
    "Adenine",
]

# ── Aliases (optional) ─────────────────────────────────────────────────
# Map label → PubChem name if the label isn't directly resolvable.

alias = {
    # "My Label": "pubchem-resolvable-name",
}

# ── Fallback queries (optional) ───────────────────────────────────────
# If the primary query fails, try these alternatives in order.

fallback_queries = {
    # "My Label": ["alt name 1", "alt name 2"],
}

# ── Expected formulas (optional) ──────────────────────────────────────
# Helps the scoring heuristic pick the right PubChem record.

expected_formulas = {
    # "Adenine": "C5H5N5",
}

---
## 2 · Step 1 — Resolve Molecules via PubChem

Queries PubChem for each molecule, scores candidate records, and writes a CSV table with CIDs, SMILES, and formulas.

In [ ]:
from pipeline.pubchem import build_molecule_table

df, diag = build_molecule_table(
    molecules,
    alias=alias,
    fallback_queries=fallback_queries,
    expected_formulas=expected_formulas,
    cache_dir=CACHE_DIR,
)

display(df)

df.to_csv(MOLECULE_CSV, index=False)
diag.to_csv(DIAG_CSV, index=False)
print(f"Wrote: {MOLECULE_CSV}")
print(f"Wrote: {DIAG_CSV}")

# Check for failures
failed = df[df["cid"].isna()][["name", "pubchem_query", "status", "warnings"]]
if len(failed):
    print("\n⚠ Molecules without CIDs (review these):")
    display(failed)

# Freeze the complete v2 run identity/configuration before writing XYZ, COM, or SH artifacts.
from pipeline.conformers import METHOD_POLICY
from pipeline.manifest import create_run_manifest, slurm_template_identity
from pipeline.slurm import DEFAULT_TEMPLATE

manifest = create_run_manifest(
    df,
    conformer={
        "method_policy": METHOD_POLICY, "seed": SEED,
        "n_generate": N_GENERATE, "top_n": TOP_N,
        "rmsd_prune": RMSD_PRUNE,
    },
    gaussian={
        "route_opt": ROUTE_OPT, "route_freq": ROUTE_FREQ,
        "title_suffix": TITLE_SUFFIX, "charge": 0,
        "multiplicity": 1, "nproc": NPROC, "link1": True,
    },
    slurm={
        "account": SLURM_ACCOUNT, "cpus": SLURM_CPUS,
        "mem": SLURM_MEM, "time": SLURM_TIME,
        "template_sha256": slurm_template_identity(DEFAULT_TEMPLATE),
    },
    path=MANIFEST_JSON,
    run_id=RUN_ID,
)
print(f"Created immutable run manifest: {MANIFEST_JSON}")

---
## 3 · Step 2 — Conformer Search (RDKit ETKDGv3 + MMFF94)

For each resolved molecule this embeds an RDKit conformer ensemble (`N_GENERATE`), RMSD-prunes duplicates, MMFF94-optimizes and ranks them, and keeps the **top `TOP_N` lowest-energy distinct** conformers as DFT starting geometries — one XYZ each, with provenance (RDKit version, seed, method, ΔE in kcal/mol) recorded in `conformer_log.csv`.

- **Rigid molecules collapse to a single conformer** on their own (e.g. adenine), so v1.1 single-geometry behavior is preserved with no special case.
- **Molecules with undefined stereochemistry are PROVISIONAL (v2.1).** Instead of being skipped, RDKit embeds **one** structure from the PubChem SMILES with an **arbitrary** choice at the undefined centre(s). It is loudly flagged: `provenance_status=provisional_undefined_stereo`, `undefined_centers`/`arbitrated_smiles` recorded, `dE=NA`, and a `PROVISIONAL: stereo arbitrated at …` marker in the XYZ/COM. It is an unvalidated DFT start, **not** the compound's real configuration — one arbitrary pick among 2^k for k undefined centres.

> Conformer energies are **MMFF94/UFF in kcal/mol** and are never mixed with the DFT Hartree energies computed later. These are force-field *starting* geometries — the DFT optimization makes the final call among the top 3.

In [ ]:
from pipeline.conformers import search_conformers

conf_log = search_conformers(
    df,
    xyz_dir=CONF_XYZ_DIR,
    log_csv=CONF_LOG_CSV,
    failed_csv=CONF_FAIL_CSV,
    n_generate=N_GENERATE,
    top_n=TOP_N,
    rmsd_prune=RMSD_PRUNE,
    seed=SEED,
    manifest_path=MANIFEST_JSON,
)
display(conf_log)

# Molecules skipped (no/undefined stereo, unparseable SMILES, or embed failure)
if os.path.exists(CONF_FAIL_CSV):
    import pandas as pd
    print("\n⚠ Skipped molecules (see conformer_search_failed.csv):")
    display(pd.read_csv(CONF_FAIL_CSV))

---
## 4 · Step 3 — Write Gaussian `.com` Files (one per conformer)

Writes one `.com` per kept conformer as `{base}_c{ii}_F.com`, with the conformer id and its ΔE (kcal/mol) in the title line. Uses the **Link1 pattern**: the optimization job writes a checkpoint, then the frequency job reads it via `Geom=AllChk Guess=Read`.

In [ ]:
from pipeline.gaussian import write_gaussian_coms_from_conformers

com_log = write_gaussian_coms_from_conformers(
    conformer_log_csv=CONF_LOG_CSV,
    outdir=GAUSSIAN_JOBS,          # COM co-located with its SH in gaussian_jobs/
    log_csv=COM_LOG_CSV,
    route_opt=ROUTE_OPT,
    route_freq=ROUTE_FREQ,
    title_suffix=TITLE_SUFFIX,
    nproc=NPROC,
    manifest_path=MANIFEST_JSON,
)
display(com_log.head())

---
## 5 · Step 4 — Generate SLURM Scripts

One `.sh` per `.com` file (i.e. one per conformer).

In [ ]:
from pipeline.slurm import write_slurm_scripts
from pipeline.manifest import finalize_manifest

# v2.1: the SH is written NEXT TO its .com in gaussian_jobs/ so the pair ships
# together. Every COM path and hash is validated before any script/log changes.
slurm_log = write_slurm_scripts(
    com_log_csv=COM_LOG_CSV,
    slurm_dir=GAUSSIAN_JOBS,       # same directory as the .com files
    log_csv=SLURM_LOG_CSV,
    manifest_path=MANIFEST_JSON,
    account=SLURM_ACCOUNT,
    cpus=SLURM_CPUS,
    mem=SLURM_MEM,
    time=SLURM_TIME,
)
finalize_manifest(MANIFEST_JSON)
display(slurm_log.head())
print(f"\nDone! Created {len(slurm_log)} SLURM scripts and verified every manifest hash.")

---
## 6 · Next Steps

1. **Archive and transfer the complete run package together:** the whole `runs/<study>/<run_id>/` folder — `run_manifest.json`, `conformer_log.csv`, `com_write_log.csv`, `slurm_write_log.csv`, `conformer_xyz/`, and `gaussian_jobs/` (COM + SH co-located). An isolated XYZ/COM/SH requires its matching manifest and is not the supported reproducibility unit.
2. **Submit** jobs **from the directory that holds each `.com`** (v2.1 operational contract). The `.sh` runs `g16 <base>_F.com` from its own working directory — the COM+SH pair ships together in `gaussian_jobs/` and the script does **not** hunt for its input, so submit from there:
   ```bash
   cd runs/<study>/<run_id>/gaussian_jobs
   for f in *.sh; do sbatch "$f"; done
   ```
3. **Check convergence** in the `.log` files after jobs complete.
4. **Parse thermochemistry** from the frequency output (ΔG, ΔH, ZPE) — remember you now have up to 3 conformers per molecule (`{base}_c00/_c01/_c02`); compare the DFT energies to pick the true minimum.
5. **Provisional (undefined-stereo) molecules** carry the `PROVISIONAL` marker and `dE=NA` through the COM. Treat their geometry as an arbitrary starting configuration, not the resolved stereoisomer, and enumerate/verify the intended stereochemistry before trusting DFT results.

> **Tip:** conformer ΔE in the `.com` title is the **MMFF force-field** ranking (kcal/mol), only a starting guess. Let the DFT optimization decide the final ordering among the carried conformers.

---
## 7 · Appendix — v1.1 legacy single-geometry path (optional)

The original v1.1 flow fed DFT a **single** Open Babel geometry per molecule (PubChem 3D SDF → Open Babel XYZ → one `.com`). The v2 conformer path above supersedes it and is the default. The cells below are **commented out** and are not part of the default walkthrough — uncomment them only if you deliberately want the legacy single-geometry behavior (requires Open Babel).

In [ ]:
# v1.1 LEGACY PATH — optional, NOT the default flow. Requires Open Babel.
# Uncomment to run the single-geometry Open Babel path instead of the conformer path.
#
# from pipeline.pubchem import download_sdfs
# from pipeline.geometry import convert_sdfs_to_xyz
# from pipeline.gaussian import write_gaussian_coms
#
# SDF_DIR = "pubchem_sdf"; XYZ_DIR = "pubchem_xyz"
# os.makedirs(SDF_DIR, exist_ok=True); os.makedirs(XYZ_DIR, exist_ok=True)
#
# sdf_log = download_sdfs(input_csv=MOLECULE_CSV, sdf_dir=SDF_DIR, log_csv="sdf_download_log.csv")
# xyz_log = convert_sdfs_to_xyz(download_log_csv="sdf_download_log.csv", xyz_dir=XYZ_DIR, log_csv="xyz_convert_log.csv")
# com_log = write_gaussian_coms(
#     xyz_log_csv="xyz_convert_log.csv", outdir=COM_DIR, log_csv="com_write_log.csv",
#     route_opt=ROUTE_OPT, route_freq=ROUTE_FREQ, title_suffix=TITLE_SUFFIX, nproc=NPROC,
# )